# Where the AUM comes from

This notebook fetches AUM **the same way the app does**, so you can check any figure yourself.

**Source:** `POST https://www.amfiindia.com/gateway/pollingsebi/api/amfi/fundperformance`
The value taken is the `dailyAUM` field, in ₹ crore.

Worth knowing before you read anything into a number:

- The service is operated by **CRISIL Research** (its error responses name
  `com.crisil.research.cfs.amfi.controller.FundPerformanceController`), hosted on AMFI's
  domain. The `pollingsebi` path segment is why this is sometimes described as SEBI data —
  but **nothing here contacts SEBI**. The dataset is AMFI's fund-performance disclosure,
  compiled under SEBI's mandate. Cite it as *AMFI, via CRISIL*.
- **There is no web page for this any more.** The old page
  (`amfiindia.com/research-information/other-data/fund-performance`) now returns 404; it
  survives in the code only as a `Referer` string. The API is the only way to see this data.
- **The feed runs about 2 days behind.** Today's AUM is never available today.
- Data starts around **2018** (SEBI recategorisation). Earlier dates return nothing.

Run the cells top to bottom. Cells 4 and 5 are the ones to edit.

## 1. Setup

Imports the project itself, so this notebook can be compared against what the app actually runs.

In [1]:
import sys, os, json, warnings
from datetime import date, timedelta

import pandas as pd
import requests

warnings.filterwarnings("ignore")

# The project imports streamlit. Outside the app that logs a harmless
# "no runtime found" warning on every import; quieten just that one.
import logging, streamlit
logging.getLogger("streamlit.runtime.caching.cache_data_api").setLevel(logging.ERROR)

PROJECT = os.getcwd()   # the notebook lives in the project folder
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 80)
print("project folder:", PROJECT)
print("today:", date.today().isoformat())

project folder: /home/sofiyaan/Desktop/NAV_Fetcher/NAV_FETCHER
today: 2026-09-26


## 2. The exact request

This is the call the app makes, written out in full so you can see every part of it.

AMFI's gateway returns **empty data** for requests that don't look like they came from its own
site, which is the usual reason AUM comes back blank. The `Origin` and `Referer` headers are what
make it answer.

In [2]:
API_URL = "https://www.amfiindia.com/gateway/pollingsebi/api/amfi/fundperformance"

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"),
    "Content-Type": "application/json",
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://www.amfiindia.com",
    "Referer": "https://www.amfiindia.com/research-information/other-data/fund-performance",
    "X-Requested-With": "XMLHttpRequest",
}

SESSION = requests.Session()


def fetch_aum(date_str, maturity_id, category_id, subcategory_id, show=False):
    """One category's AUM for one day. date_str looks like '24-Sep-2026'.

    Returns the raw list of scheme rows, exactly as AMFI sends them.
    """
    payload = {
        "maturityType": maturity_id,
        "category": category_id,
        "subCategory": subcategory_id,
        "mfid": 0,
        "reportDate": date_str,
    }
    if show:
        print("POST", API_URL)
        print("payload:", json.dumps(payload))
    r = SESSION.post(API_URL, json=payload, headers=HEADERS, timeout=(10, 40))
    r.raise_for_status()
    body = r.json()
    rows = body.get("data") or []
    if show:
        print(f"HTTP {r.status_code} | validationMsg={body.get('validationMsg')} | rows={len(rows)}")
    return rows


print("ready")

ready


## 3. Which category are you asking for?

The API wants three numeric ids, not a fund name. The project resolves them from the AMFI section
heading with `map_section_to_ids`, and this uses that same function so the ids can't drift apart.

Change `SECTION` to query a different category.

In [3]:
from nav_fetcher import map_section_to_ids

SECTION = "Open Ended Schemes ( Equity Scheme - Large Cap Fund )"   # <-- edit me

MATURITY, CATEGORY, SUBCATEGORY = map_section_to_ids(SECTION)
print(f"{SECTION}\n  -> maturityType={MATURITY}, category={CATEGORY}, subCategory={SUBCATEGORY}")

# A few others you can paste into SECTION above:
for s in [
    "Open Ended Schemes ( Equity Scheme - Mid Cap Fund )",
    "Open Ended Schemes ( Equity Scheme - Small Cap Fund )",
    "Open Ended Schemes ( Equity Scheme - Flexi Cap Fund )",
    "Open Ended Schemes ( Hybrid Scheme - Aggressive Hybrid Fund )",
    "Open Ended Schemes ( Debt Scheme - Liquid Fund )",
]:
    print(f"  {map_section_to_ids(s)}   {s}")

Open Ended Schemes ( Equity Scheme - Large Cap Fund )
  -> maturityType=1, category=1, subCategory=1
  (1, 1, 5)   Open Ended Schemes ( Equity Scheme - Mid Cap Fund )
  (1, 1, 6)   Open Ended Schemes ( Equity Scheme - Small Cap Fund )
  (1, 1, 3)   Open Ended Schemes ( Equity Scheme - Flexi Cap Fund )
  (1, 3, 30)   Open Ended Schemes ( Hybrid Scheme - Aggressive Hybrid Fund )
  (1, 2, 20)   Open Ended Schemes ( Debt Scheme - Liquid Fund )


## 4. One day's AUM for the whole category

Every scheme AMFI published for that category on that date. `dailyAUM` is in ₹ crore.

In [4]:
REPORT_DATE = "24-Sep-2026"          # <-- edit me ('DD-Mon-YYYY'; remember the ~2 day lag)

rows = fetch_aum(REPORT_DATE, MATURITY, CATEGORY, SUBCATEGORY, show=True)

if not rows:
    print("\nNo data for that date. It may be a weekend, a holiday, or too recent (2-day lag).")
else:
    df = pd.DataFrame(rows)[["schemeName", "dailyAUM", "navDate", "navRegular", "benchmark"]]
    df["dailyAUM"] = pd.to_numeric(df["dailyAUM"], errors="coerce")
    display(df.sort_values("dailyAUM", ascending=False).reset_index(drop=True))

POST https://www.amfiindia.com/gateway/pollingsebi/api/amfi/fundperformance
payload: {"maturityType": 1, "category": 1, "subCategory": 1, "mfid": 0, "reportDate": "24-Sep-2026"}
HTTP 200 | validationMsg=SUCCESS | rows=35


,schemeName,dailyAUM,navDate,navRegular,benchmark
0,ICICI Prudential Large Cap Fund,77282.460000,24-Sep-2026,104.1900,Nifty 100 TRI
1,SBI Large Cap Fund,53049.380000,24-Sep-2026,90.3696,BSE 100 TRI
2,Nippon India Large Cap Fund,52187.080000,24-Sep-2026,86.5191,BSE 100 TRI
3,HDFC Large Cap Fund,38113.040000,24-Sep-2026,1090.4020,Nifty 100 TRI
4,Mirae Asset Large Cap Fund,36297.266000,24-Sep-2026,107.9770,Nifty 100 TRI
5,Axis Large Cap Fund,30366.496000,24-Sep-2026,58.2100,BSE 100 TRI
6,Aditya Birla Sun Life Large Cap Fund,28414.613000,24-Sep-2026,501.3400,Nifty 100 TRI
7,Canara Robeco Large Cap Fund,16274.280000,24-Sep-2026,59.4900,BSE 100 TRI
8,UTI Large Cap Fund,11636.416000,24-Sep-2026,255.7542,BSE 100 TRI
9,Kotak Large Cap Fund,10890.354000,24-Sep-2026,545.1940,Nifty 100 TRI


## 5. Is it actually updating, or a frozen number?

Pick one scheme and pull it across several dates. If the figures move, the feed is live.

Copy a name **exactly** as it appears in the `schemeName` column above.

In [5]:
SCHEME = "Aditya Birla Sun Life Large Cap Fund"     # <-- edit me

DATES = ["01-Sep-2026", "08-Sep-2026", "15-Sep-2026",
         "22-Sep-2026", "23-Sep-2026", "24-Sep-2026"]      # <-- edit me

records = []
for d in DATES:
    value = None
    for r in fetch_aum(d, MATURITY, CATEGORY, SUBCATEGORY):
        if str(r.get("schemeName", "")).strip() == SCHEME:
            value = pd.to_numeric(r.get("dailyAUM"), errors="coerce")
            break
    records.append({"date": d, "dailyAUM (Rs cr)": value})

series = pd.DataFrame(records)
display(series)

vals = series["dailyAUM (Rs cr)"].dropna()
print(f"{len(vals)} values fetched, {vals.nunique()} of them distinct")
print("Moving day to day:", vals.nunique() > 1)
if len(vals) > 1:
    print(f"Range: {vals.min():,.2f} to {vals.max():,.2f}  "
          f"(swing of {vals.max() - vals.min():,.2f} cr)")

,date,dailyAUM (Rs cr)
0,01-Sep-2026,29578.246
1,08-Sep-2026,29079.814
2,15-Sep-2026,28358.777
3,22-Sep-2026,28720.895
4,23-Sep-2026,28876.611
5,24-Sep-2026,28414.613


6 values fetched, 6 of them distinct
Moving day to day: True
Range: 28,358.78 to 29,578.25  (swing of 1,219.47 cr)


## 6. Does this match what the app uses?

The cell above talks to AMFI directly. This one calls the project's own function for the same
day and compares. If these disagree, the notebook and the app are not measuring the same thing.

In [6]:
import nav_fetcher

CHECK_DATE = REPORT_DATE

mine = {str(r.get("schemeName", "")).strip(): r.get("dailyAUM")
        for r in fetch_aum(CHECK_DATE, MATURITY, CATEGORY, SUBCATEGORY)}
theirs = {str(r.get("schemeName", "")).strip(): r.get("dailyAUM")
          for r in nav_fetcher.fetch_performance_data_from_api(
              CHECK_DATE, MATURITY, CATEGORY, SUBCATEGORY)}

print(f"notebook: {len(mine)} schemes | app: {len(theirs)} schemes")
disagree = {k: (mine.get(k), theirs.get(k))
            for k in set(mine) | set(theirs) if mine.get(k) != theirs.get(k)}
print("Identical:", not disagree)
if disagree:
    display(pd.DataFrame(
        [{"scheme": k, "notebook": v[0], "app": v[1]} for k, v in disagree.items()]))

notebook: 35 schemes | app: 35 schemes
Identical: True


## 7. How fresh is the feed right now?

Walks back from today until a date returns data. That gap is the publication lag, and it is why
an MIS run today cannot show today's AUM.

In [7]:
today = date.today()
for i in range(10):
    ds = (today - timedelta(days=i)).strftime("%d-%b-%Y")
    n = len(fetch_aum(ds, MATURITY, CATEGORY, SUBCATEGORY))
    print(f"  {ds}: {n} rows")
    if n:
        print(f"\nLatest published date: {ds}  ({i} day(s) behind today)")
        break
else:
    print("\nNothing in the last 10 days -- the feed may be down.")

  26-Sep-2026: 0 rows
  25-Sep-2026: 0 rows
  24-Sep-2026: 35 rows

Latest published date: 24-Sep-2026  (2 day(s) behind today)


## 8. When the API has nothing: the fallback

If the API can't answer for a scheme and date, the app does **not** leave the figure blank. It
falls back to a month-end AUM from `scheme_aum_monthly.xlsx` (a local file, committed to the repo,
not fetched from anywhere) and scales it by that day's NAV relative to the scheme's mean NAV, so
the number varies day to day.

That means **a daily AUM in a report is one of two quite different things**: a real published
figure, or a month-end number stretched across days. This cell shows what that file currently
covers.

In [8]:
path = os.path.join(PROJECT, "scheme_aum_monthly.xlsx")
if not os.path.exists(path):
    print("scheme_aum_monthly.xlsx not found -- the fallback would be unavailable.")
else:
    fb = pd.read_excel(path, header=3)
    fb["PD_Month End"] = pd.to_numeric(fb["PD_Month End"], errors="coerce")
    months = sorted(fb["PD_Month End"].dropna().astype(int).unique())
    print(f"rows: {len(fb)} | distinct schemes (ISIN): {fb['SD_Scheme ISIN'].nunique()}")
    print(f"months covered: {months[0]} to {months[-1]}  (YYYYMM)")
    newest = str(months[-1])
    print(f"\nNewest month-end on file: {newest[:4]}-{newest[4:]}")
    print("Anything after that date falls back to this month-end, scaled by NAV.")
    display(fb.head())

rows: 174 | distinct schemes (ISIN): 29
months covered: 202511 to 202604  (YYYYMM)

Newest month-end on file: 2026-04
Anything after that date falls back to this month-end, scaled by NAV.


,SD_Scheme ISIN,PD_Month End,PD_Scheme AUM
0,INF209K01AJ8,202602,25206.6038
1,INF209K01AJ8,202603,23018.3400
2,INF209K01AJ8,202601,24700.4305
3,INF209K01AJ8,202604,25631.5095
4,INF209K01AJ8,202511,24815.1052


## What to cite

> Scheme AUM: AMFI fund-performance disclosure (`dailyAUM`), retrieved from
> `amfiindia.com/gateway/pollingsebi/api/amfi/fundperformance`, operated by CRISIL Research.
> Published with an approximately 2-day lag; series begins 2018.

Two things to keep in mind when reconciling a report:

1. **quant AMC files a day out of step.** The app pulls their AUM series back one observation;
   their NAVs are treated like everyone else's. So a quant figure here will not line up with a
   quant figure in a report unless you apply the same shift.
2. **A figure may be the fallback, not a published number.** Cell 8 shows when that kicks in.